# Project 1: Data Analysis 

## 1 Inequality in Denmark

In [ ]:
# -------------------------------------------------
# Imports and data
# -------------------------------------------------

# Imprts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Indlæser og sortere data
url41 = "https://api.statbank.dk/v1/data/IFOR41/CSV?ULLIG=70&KOMMUNEDK=*&Tid=*"
gini_raw = pd.read_csv(url41, sep=";")

gini_raw = gini_raw[["KOMMUNEDK", "TID", "INDHOLD"]].copy()
gini_raw.columns = ["municipality", "Year", "Gini"]

gini_raw["municipality"] = gini_raw["municipality"].astype(str)
gini_raw["Year"] = pd.to_numeric(gini_raw["Year"], errors="coerce")
gini_raw["Gini"] = pd.to_numeric(
    gini_raw["Gini"].astype(str).str.replace(",", "."),
    errors="coerce"
)
gini_raw = gini_raw.dropna().sort_values(["municipality", "Year"]).reset_index(drop=True)

url32 = "https://api.statbank.dk/v1/data/IFOR32/CSV?DECILGEN=*&KOMMUNEDK=*&Tid=*"
decile_raw = pd.read_csv(url32, sep=";")

decile_raw = decile_raw[["KOMMUNEDK", "DECILGEN", "TID", "INDHOLD"]].copy()
decile_raw.columns = ["municipality", "decile", "Year", "income"]

decile_raw["municipality"] = decile_raw["municipality"].astype(str)
decile_raw["Year"] = pd.to_numeric(decile_raw["Year"], errors="coerce")
decile_raw["income"] = pd.to_numeric(
    decile_raw["income"].astype(str).str.replace(",", "."),
    errors="coerce"
)
decile_raw["decile"] = decile_raw["decile"].astype(str).str.extract(r"(\d+)")
decile_raw["decile"] = pd.to_numeric(decile_raw["decile"], errors="coerce")
decile_raw = decile_raw.dropna().sort_values(["municipality", "Year", "decile"]).reset_index(drop=True)


# -------------------------------------------------
# 1.1: Gini + Top 10% share (national data)
# -------------------------------------------------

# National data: keep only "Hele landet"
gini_nat = gini_raw[
    gini_raw["municipality"].str.contains("Hele landet", case=False, na=False)
].copy().sort_values("Year").reset_index(drop=True)

decile_nat = decile_raw[
    decile_raw["municipality"].str.contains("Hele landet", case=False, na=False)
].copy().sort_values(["Year", "decile"]).reset_index(drop=True)

decile_pivot = decile_nat.pivot_table(
    index="Year",
    columns="decile",
    values="income",
    aggfunc="first"
).sort_index()

decile_pivot = decile_pivot.reindex(columns=range(1, 11), fill_value=np.nan)
top10_share = decile_pivot[10] / decile_pivot.sum(axis=1)

top10 = top10_share.reset_index()
top10.columns = ["Year", "Top10Share"]

merged = gini_nat.merge(top10, on="Year", how="inner")

# Figure 1: Gini coefficient over time
plt.figure(figsize=(10, 6))
plt.plot(merged["Year"], merged["Gini"], color="blue", marker="o")
plt.title("Gini coefficient in Denmark")
plt.xlabel("Year")
plt.ylabel("Gini coefficient")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Figure 2: Gini and top 10% share in the same figure
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(merged["Year"], merged["Gini"], color="blue", marker="o", label="Gini coefficient")
ax1.set_xlabel("Year")
ax1.set_ylabel("Gini coefficient", color="blue")
ax1.tick_params(axis="y", labelcolor="blue")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(merged["Year"], merged["Top10Share"] * 100, color="red", marker="s", label="Top 10% share")
ax2.set_ylabel("Top 10% share (%)", color="red")
ax2.tick_params(axis="y", labelcolor="red")

plt.title("Gini coefficient and Top 10% income share in Denmark")
fig.tight_layout()
plt.show()

corr = merged["Gini"].corr(merged["Top10Share"])
print(f"Correlation: {corr:.3f}")

# -------------------------------------------------
# 1.2: Prediction
# -------------------------------------------------

# Minimize SSE with scipy.optimize.minimize
gini_fit = gini_nat.copy()
gini_fit["t"] = gini_fit["Year"] - gini_fit["Year"].min()

t = gini_fit["t"].to_numpy()
y = gini_fit["Gini"].to_numpy()

# Linear trend
def sse_linear(beta, t, y):
    y_hat = beta[0] + beta[1] * t
    return np.sum((y - y_hat) ** 2)

result_linear = minimize(
    sse_linear,
    x0=[20, 0.1],
    args=(t, y)
)

beta_linear = result_linear.x
print("Linear coefficients:")
print(f"beta0 = {beta_linear[0]:.4f}")
print(f"beta1 = {beta_linear[1]:.4f}")

# Quadratic trend
def sse_quadratic(beta, t, y):
    y_hat = beta[0] + beta[1] * t + beta[2] * t**2
    return np.sum((y - y_hat) ** 2)

result_quadratic = minimize(
    sse_quadratic,
    x0=[20, 0.1, 0.001],
    args=(t, y)
)

beta_quadratic = result_quadratic.x
print("Quadratic coefficients:")
print(f"beta0 = {beta_quadratic[0]:.4f}")
print(f"beta1 = {beta_quadratic[1]:.4f}")
print(f"beta2 = {beta_quadratic[2]:.6f}")

# Minimize with polyfit
coef_lin = np.polyfit(t, y, 1)
coef_quad = np.polyfit(t, y, 2)

print(f"Linear coefficients: {coef_lin[0]:.3f}, {coef_lin[1]:.3f}")
print(f"Quadratic coefficients: {coef_quad[0]:.3f}, {coef_quad[1]:.3f}, {coef_quad[2]:.3f}")

# Forecasts for 2030 and 2040 with a for loop
for year in [2030, 2040]:
    tau = year - gini_fit["Year"].min()
    lin_pred = beta_linear[0] + beta_linear[1] * tau
    quad_pred = beta_quadratic[0] + beta_quadratic[1] * tau + beta_quadratic[2] * tau**2
    print(f"Year {year}: linear = {lin_pred:.3f}, quadratic = {quad_pred:.3f}")

# Plot observed data + fitted trends
years_future = np.arange(gini_fit["Year"].min(), 2041)
t_future = years_future - gini_fit["Year"].min()

linear_fit = beta_linear[0] + beta_linear[1] * t_future
quadratic_fit = beta_quadratic[0] + beta_quadratic[1] * t_future + beta_quadratic[2] * t_future**2

plt.figure(figsize=(10, 6))
plt.scatter(gini_fit["Year"], gini_fit["Gini"], color="black", label="Observed Gini")
plt.plot(years_future, linear_fit, color="blue", linewidth=2, label="Linear trend")
plt.plot(years_future, quadratic_fit, color="red", linewidth=2, label="Quadratic trend")
plt.axvline(2030, color="gray", linestyle="--", alpha=0.7)
plt.axvline(2040, color="gray", linestyle="--", alpha=0.7)
plt.title("Gini coefficient: historical data and forecasts")
plt.xlabel("Year")
plt.ylabel("Gini")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# -------------------------------------------------
# 1.3: Municipalities
# -------------------------------------------------

# Reuse the already cleaned raw data and filter out the national aggregate
gini = gini_raw[
    ~gini_raw["municipality"].str.contains("Hele landet", case=False, na=False)
].copy().sort_values(["municipality", "Year"]).reset_index(drop=True)

decile = decile_raw[
    ~decile_raw["municipality"].str.contains("Hele landet", case=False, na=False)
].copy().sort_values(["municipality", "Year", "decile"]).reset_index(drop=True)

# 2) Top 10% share
decile_pivot = decile.pivot_table(
    index=["municipality", "Year"],
    columns="decile",
    values="income",
    aggfunc="first"
).reindex(columns=range(1, 11), fill_value=np.nan)

top10_share = decile_pivot[10] / decile_pivot.sum(axis=1)
top10 = top10_share.reset_index()
top10.columns = ["municipality", "Year", "Top10Share"]

# 3) Latest year and correlation across municipalities
latest_year = max(gini["Year"].max(), top10["Year"].max())
gini_latest = gini[gini["Year"] == latest_year].copy()
top10_latest = top10[top10["Year"] == latest_year].copy()

df = gini_latest.merge(top10_latest, on=["municipality", "Year"], how="inner")

corr = df["Gini"].corr(df["Top10Share"])
print(f"Correlation across municipalities in {latest_year}: {corr:.3f}")

# 4) Most unequal / most equal municipalities in the most recent year
most_unequal = df.sort_values("Gini", ascending=False).head(10).copy()
most_equal = df.sort_values("Gini", ascending=True).head(10).copy()

print("Most unequal:")
print(most_unequal[["municipality", "Gini"]])

print("\nMost equal:")
print(most_equal[["municipality", "Gini"]])

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].bar(most_unequal["municipality"], most_unequal["Gini"], color="darkred")
axes[0].set_title(f"10 most unequal municipalities in {latest_year}")
axes[0].set_ylabel("Gini")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(most_equal["municipality"], most_equal["Gini"], color="darkgreen")
axes[1].set_title(f"10 most equal municipalities in {latest_year}")
axes[1].set_ylabel("Gini")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# 5) Largest and smallest change in Gini over time
gini_change = (
    gini.groupby("municipality")
    .agg(gini_first=("Gini", "first"), gini_last=("Gini", "last"))
    .reset_index()
)
gini_change["change"] = gini_change["gini_last"] - gini_change["gini_first"]

largest_change = gini_change.sort_values("change", ascending=False).head(10)
smallest_change = gini_change.sort_values("change", ascending=True).head(10)

print("\nLargest increase in Gini:")
print(largest_change[["municipality", "gini_first", "gini_last", "change"]])

print("\nSmallest change in Gini:")
print(smallest_change[["municipality", "gini_first", "gini_last", "change"]])

fig, axes = plt.subplots(2, 1, figsize=(12, 10))

axes[0].bar(largest_change["municipality"], largest_change["change"], color="royalblue")
axes[0].set_title("10 municipalities with the largest increase in Gini")
axes[0].set_ylabel("Change in Gini")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(smallest_change["municipality"], smallest_change["change"], color="darkorange")
axes[1].set_title("10 municipalities with the smallest change in Gini")
axes[1].set_ylabel("Change in Gini")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

# -------------------------------------------------
# 1.4: Extensions
# -------------------------------------------------

# Also inspect the underlying series for the municipalities with the largest/smallest changes
selected_change_municipalities = list(largest_change["municipality"].head(3)) + list(smallest_change["municipality"].head(3))
change_series_selected = gini[gini["municipality"].isin(selected_change_municipalities)].copy()

fig, ax = plt.subplots(figsize=(12, 7))
for muni, grp in change_series_selected.groupby("municipality"):
    ax.plot(grp["Year"], grp["Gini"], marker="o", linewidth=2, label=muni)
ax.set_title("Underlying Gini series for municipalities with extreme changes")
ax.set_xlabel("Year")
ax.set_ylabel("Gini")
ax.grid(True, alpha=0.3)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

print("\nInterpretation: the municipality-level correlation is very strong and positive,\nwhich matches the national pattern over time. The unequal municipalities are mainly\nwealthy suburban municipalities, while the more equal ones tend to be smaller or\nless affluent municipalities. The largest changes mostly reflect municipal changes\nover time rather than one-off extreme incomes, so the time-series plots are important\nfor checking whether a ranking is driven by a single unusually large income year.")


## 2 Simulating the income distribution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib.util
import pathlib
import sys

# Load the original model from the existing file
src_path = pathlib.Path('model 2.1.py')
spec = importlib.util.spec_from_file_location('model_2_1', str(src_path))
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
simulate_income_model = module.simulate_income_model

# ---------------------------------------------------------------------
# 2.2 Simulate the income distribution
# ---------------------------------------------------------------------

sim = simulate_income_model(seed=1234)
ages = sim['ages']
income = sim['income']
education = sim['education']
employed = sim['employed']
labor_force = sim['labor_force']

# Vores checks
# Check shape of income array
print("Income shape:", income.shape)

# Check education shares
for e in range(3):
    share = np.mean(education == e)
    print(e, share)

# Check unemployment rate
unemployed = labor_force & ~employed

unemployment_rate = (
    unemployed.sum(axis=0)
    / labor_force.sum(axis=0)
)

steady_state = 0.05 / (0.05 + 0.60)

print("Unemployment rate at age 65:", unemployment_rate[-1])
print("Theoretical steady state:", steady_state)


job_finding = 0.60
job_separation = 0.05
steady_state_u = job_separation / (job_separation + job_finding)
print(f'Theoretical steady-state unemployment rate: {steady_state_u:.3f}')
print(f'Average unemployment rate for ages >= 25: {np.nanmean(unemployment_by_age[ages >= 25]):.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ages, unemployment_by_age, color='black', linewidth=2)
ax.axhline(steady_state_u, color='red', linestyle='--', linewidth=1.5, label='steady state')
ax.set_title('Unemployment rate over the life cycle')
ax.set_xlabel('Age')
ax.set_ylabel('Unemployment rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2.2.2 Plot mean and selected percentiles of income over the life cycle
mean_income = income.mean(axis=0)
percentiles = [10, 25, 50, 75, 90]
percentile_income = np.percentile(income, percentiles, axis=0)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ages, mean_income, color='navy', linewidth=2.5, label='Mean income')
for p, values in zip(percentiles, percentile_income):
    ax.plot(ages, values, linestyle='--', alpha=0.8, label=f'{p}th percentile')
ax.set_title('Income over the life cycle')
ax.set_xlabel('Age')
ax.set_ylabel('Income')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2.2.3 Histogram of the income distribution at selected ages
selected_ages = [25, 35, 45, 60]
selected_idx = [np.argmin(np.abs(ages - age)) for age in selected_ages]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for ax, age, idx in zip(axes, selected_ages, selected_idx):
    income_at_age = income[:, idx]
    ax.hist(income_at_age, bins=40, density=True, alpha=0.7, color='steelblue')
    ax.set_title(f'Income distribution at age {age}')
    ax.set_xlabel('Income')
    ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('- The education shares match the target probabilities, so the initial composition is correctly calibrated.')
print('- The unemployment rate settles near the theoretical steady-state value s / (s + f).')
print('- Income rises over the life cycle and the right tail becomes thicker at older ages, so the distribution becomes more dispersed and more skewed.')

# ---------------------------------------------------------------------
# 2.3 Compute the Gini coefficient and plot the Lorenz curve
# ---------------------------------------------------------------------

def gini(x):
    x = np.asarray(x, dtype=float).ravel()
    if x.size == 0:
        return np.nan
    if np.all(x == 0):
        return 0.0
    x = np.sort(x)
    n = x.size
    idx = np.arange(1, n + 1)
    return float(2 * np.sum(idx * x) / (n * np.sum(x)) - (n + 1) / n)

# Check the function on a known case
uniform_draws = np.linspace(0, 1, 10000)
print(f'Gini of uniform [0,1]: {gini(uniform_draws):.6f}')

pooled_gini = gini(income.ravel())
print(f'Gini coefficient for the full simulated sample: {pooled_gini:.3f}')

sorted_income = np.sort(income.ravel())
population_share = np.linspace(0, 1, len(sorted_income))
income_share = np.cumsum(sorted_income) / np.sum(sorted_income)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(population_share, population_share, color='black', linestyle='--', label='45-degree line')
ax.plot(population_share, income_share, color='darkred', linewidth=2, label='Lorenz curve')
ax.fill_between(population_share, population_share, income_share, alpha=0.2, color='darkred')
ax.set_title('Lorenz curve for the full simulated sample')
ax.set_xlabel('Population share')
ax.set_ylabel('Income share')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compare pooled inequality and age-specific inequality
age_gini = {}
for age in [25, 35, 45, 60]:
    idx = np.argmin(np.abs(ages - age))
    age_gini[age] = gini(income[:, idx])

print('Gini by age:')
for age in [25, 35, 45, 60]:
    print(f'  Age {age}: {age_gini[age]:.3f}')

print(f'Pooled Gini: {pooled_gini:.3f}')
print('Interpretation: inequality is usually lower within a single age group than in the pooled sample, because pooling adds age differences in human capital and labor-market status.')

# ---------------------------------------------------------------------
# 2.4 What drives inequality?
# ---------------------------------------------------------------------
# We keep the original model file as the baseline and interpret the main
# mechanisms qualitatively from the model structure.

print('\nWhat drives inequality?')
print('- Educational differences create differences in initial human capital and schooling length.')
print('- Human-capital shocks widen the income distribution through multiplicative productivity shocks.')
print('- Depreciation while unemployed erodes human capital and makes re-employment more difficult.')
print('- Unemployment directly reduces income and creates a mass of lower-income households.')
print('The baseline model therefore generates inequality through a combination of labor-market risk and heterogeneous human capital accumulation.')

# ---------------------------------------------------------------------
# 2.5 Extension: more risk
# ---------------------------------------------------------------------
# This extension adds an extra multiplicative shock to the baseline model output.
extra_risk_sd = 0.30
rng = np.random.default_rng(2024)
extra_shock = rng.lognormal(mean=-0.5 * extra_risk_sd**2, sigma=extra_risk_sd, size=income.shape)
extended_income = income * extra_shock

baseline_gini = gini(income.ravel())
extended_gini = gini(extended_income.ravel())
print(f'\nExtra-risk extension: baseline pooled Gini = {baseline_gini:.3f}; extended pooled Gini = {extended_gini:.3f}')

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(income.ravel(), bins=60, density=True, alpha=0.5, color='blue', label='baseline')
ax.hist(extended_income.ravel(), bins=60, density=True, alpha=0.5, color='red', label='with extra risk')
ax.set_title('Effect of an additional risk shock on the income distribution')
ax.set_xlabel('Income')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Interpretation: a higher risk parameter creates a thicker right tail and raises the Gini coefficient, because income becomes more dispersed across individuals.')
